# Class 3 - Regularization and cross-validation

In [ ]:
# %%capture
# !pip install matplotlib numpy pandas seaborn scikit-learn tqdm

In [ ]:
import warnings
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm

warnings.filterwarnings("ignore", message=".*deprecated")

In [ ]:
plt.style.use("bmh")

## Dataset - preparation and One-Hot Encoding

In [ ]:
# Titanic dataset - information about passengers with an indication of survival
# Task: binary classification of target column Survived
dataset = pd.read_csv(
    "https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv"
)

**Dataset exploration**

In [ ]:
dataset.isna().sum()

In [ ]:
dataset.describe()

In [ ]:
dataset.describe(exclude="number")

In [ ]:
dataset["Parents/Children Aboard"].value_counts().sort_index()

In [ ]:
dataset.Pclass.value_counts()

In [ ]:
dataset.Survived.value_counts(normalize=True)

**Dataset preprocessing**

In [ ]:
dataset = dataset.drop(columns="Name")

In [ ]:
dataset.Pclass = dataset.Pclass.astype(str)

In [ ]:
dataset = pd.get_dummies(dataset, drop_first=True)

In [ ]:
dataset

In [ ]:
X = dataset.drop(columns="Survived")
y = dataset.Survived
X_train, X_val, y_train, y_val = train_test_split(
    X, y, train_size=0.6, stratify=y, random_state=42
)

**Feature scaling**

Regularization penalizes all coefficients equally, so its effect depends on the scale of the features. Therefore, we standardize all columns. The scaler is fitted on the training set only and then applied to the validation set - fitting it on the whole dataset would leak information from the validation set into training.

In [ ]:
scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

## Regularization

**Regularization** techniques are modifications introduced in models aimed at **reducing overfitting**. The modification usually involves putting constraints on the coefficient estimates or altering the training process with additional steps. A good regularization technique decreases **variance** significantly, while not increasing **bias** or increasing it only slightly.

Examples of regularization techniques are:
- L1 regularization (e.g. lasso regression)
- L2 regularization (e.g. ridge regression)
- elastic net (combination of L1 and L2)
- cost-complexity pruning for decision trees
- cost parameter C for support vector machines
- dropout for neural networks
- early stopping

L1 (lasso):$$\min_w\frac{1}{n}||Xw-y||_2^2+\lambda||w||_1 \quad \left(||w||_1=\sum_i|w_i|\right)$$
L2 (ridge):$$\min_w\frac{1}{n}||Xw-y||_2^2+\lambda||w||_2^2 \quad \left(||w||_2^2=\sum_iw_i^2\right)$$
Elastic net:
$$\min_w\frac{1}{n}||Xw-y||_2^2+\lambda\left(\alpha||w||_1+(1-\alpha)||w||_2^2\right)$$

The formulas above use the squared error loss (linear regression). For logistic regression the squared error is replaced by the log-loss, while the penalty terms stay the same.

**Notation in scikit-learn:** `LogisticRegression` and `SVC` use the parameter `C`, which is the **inverse** of the regularization strength ($C \sim 1/\lambda$), so **smaller `C` means stronger regularization**. The elastic net mixing parameter $\alpha$ is called `l1_ratio` in `LogisticRegression`.

In [ ]:
lr_titanic = LR(penalty=None, max_iter=1000)
lr_titanic.fit(X_train, y_train)
lr_no_reg_auc = roc_auc_score(y_val, lr_titanic.predict_proba(X_val)[:, 1])
print(f"AUC for logistic regression with no regularization: {lr_no_reg_auc:.3f}")

**Hyperparameter tuning**

Tuning elements:
- metric (F1-score, AUC)
- hyperparameter(s) (cutoff threshold, regularization strength, number of trees)
- technique (grid search, random search, Bayesian optimization)

Task: Find the value of regularization strength (`C`) and the type of regularization (L1/L2) that maximize AUC on the validation set

In [ ]:
def model_auc(model, X_train, X_val, y_train, y_val):
    trained_model = model.fit(X_train, y_train)
    return roc_auc_score(y_val, trained_model.predict_proba(X_val)[:,1])

Logistic regression with L1 regularization

In [ ]:
# 200 values of C evenly distributed between 0.01 and 3
cs = np.linspace(0.01, 3.0, 200)

In [ ]:
LR_L1 = partial(
    LR, penalty="l1", max_iter=1000, solver="liblinear", random_state=42
)
aucs_l1 = [model_auc(LR_L1(C=c), X_train, X_val, y_train, y_val) for c in tqdm(cs)]

In [ ]:
p = sns.lineplot(x=cs, y=aucs_l1)
p.set_xlabel("C")
p.set_ylabel("AUC")
p.set_title("Logistic regression with L1 penalty");

In [ ]:
# Coefficients for standardized features
def lr_coeffs_for_c(model, c):
    model = model(C=c).fit(X_train, y_train)
    rounded_coefs = [round(e.item(), 3) for e in model.coef_[0]]
    return dict(zip(X_train.columns, rounded_coefs))

In [ ]:
lr_coeffs_for_c(LR_L1, 0.01)

In [ ]:
lr_coeffs_for_c(LR_L1, 0.1)

**Important note:** L1 penalty reduces overfitting, but it also serves as a **feature selection** method - the stronger the regularization (the smaller `C`), the more coefficients are set exactly to zero.

Logistic regression with L2 regularization

In [ ]:
aucs_l2 = [
    model_auc(LR(C=c, max_iter=1000), X_train, X_val, y_train, y_val)
    for c in tqdm(cs)
]

In [ ]:
p = sns.lineplot(x=cs, y=aucs_l2)
p.set_xlabel("C")
p.set_ylabel("AUC")
p.set_title("Logistic regression with L2 penalty");

In [ ]:
lr_coeffs_for_c(LR, 0.0001)

Support Vector Classifier

In [ ]:
cs_svc = np.linspace(0.01, 1, 100)
aucs_svc = [
    model_auc(SVC(C=c, probability=True), X_train, X_val, y_train, y_val) for c in tqdm(cs_svc)
]

In [ ]:
p = sns.lineplot(x=cs_svc, y=aucs_svc)
p.set_xlabel("C")
p.set_ylabel("AUC")
p.set_title("Support Vector Classifier");

**Summary for all models**

In [ ]:
print(f"""Logistic regression (no penalty): {lr_no_reg_auc:.4f} AUC
Logistic regression (L1): {max(aucs_l1):.4f} AUC for C={cs[np.argmax(aucs_l1)]:.4f}
Logistic regression (L2): {max(aucs_l2):.4f} AUC for C={cs[np.argmax(aucs_l2)]:.4f}
SVC: {max(aucs_svc):.4f} AUC for C={cs_svc[np.argmax(aucs_svc)]:.4f}""")

**Note:** the best AUC values above are **optimistically biased** - the validation set was used to choose `C`, so it is no longer an independent estimate of model performance. The final performance of the chosen model should be evaluated on a separate test set that was not used during tuning (see the scheme in the cross-validation section).

## Cross-validation

<img src="https://scikit-learn.org/stable/_images/grid_search_cross_validation.png" width=75%>

Advantages:
- reduces the variance of the performance estimate caused by a single random train/validation split
- every observation is used for both training and validation
- provides information about the variability of the score (standard deviation across folds)
- no need to set aside a separate validation subset

Disadvantages:
- computationally expensive (training _k_ models instead of 1)
- introduces another hyperparameter (_k_)
- more complex training and evaluation pipeline (e.g. preprocessing such as scaling must be fitted inside each fold to avoid data leakage)

More information can be found on the [cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) page in the scikit-learn documentation.

In [ ]:
def auc_scorer(model, X, y):
    return roc_auc_score(y, model.predict_proba(X)[:,1])

Cross-validation splits the whole (unscaled) `X` into folds, so the scaler has to be fitted separately in each fold on its training part. Putting the scaler and the model together in a pipeline (`make_pipeline`) does exactly that.

In [ ]:
folds = 3
scores = cross_val_score(
    make_pipeline(StandardScaler(), LR(max_iter=1000, random_state=42)),
    X,
    y,
    cv=folds,
    scoring=auc_scorer,
)  # scoring='roc_auc'
print(scores)
print(f"Mean AUC score: {np.mean(scores):.3f}")

In [ ]:
aucs_mean = np.array([])
aucs_std = np.array([])
cs_svc_xval = np.linspace(0.01, 2, 50)
for c in tqdm(cs_svc_xval):
    xval_arr = cross_val_score(
        make_pipeline(StandardScaler(), SVC(C=c, probability=True)), X, y, cv=folds, scoring=auc_scorer
    )
    aucs_mean = np.append(aucs_mean, np.mean(xval_arr))
    aucs_std = np.append(aucs_std, np.std(xval_arr))

In [ ]:
plt.plot(cs_svc_xval, aucs_mean, "r")
plt.fill_between(
    cs_svc_xval, aucs_mean - aucs_std, aucs_mean + aucs_std, color="steelblue"
)
best_c = cs_svc_xval[np.argmax(aucs_mean)]
best_mean_auc = np.max(aucs_mean)
plt.plot(best_c, best_mean_auc, "bo")
plt.annotate(
    f"AUC: {best_mean_auc:.3f} \nC: {best_c:.3f}",
    (best_c, best_mean_auc * 0.98),
    weight="bold",
)
plt.xlabel("C")
plt.ylabel("Mean AUC ± 1 Std")
plt.title(f"Support Vector Classifier - {folds}-fold Xval");

Grid search hyperparameter tuning + cross-validation = [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

### Homework (5 pts, deadline: 19.01.2027)

Perform hyperparameter tuning on the prepared **Titanic dataset** (`X` and `y`) using:
1. `GridSearchCV`
2. `RandomizedSearchCV`

Tune hyperparameters of `LogisticRegression` as follows:
- target metric: F1-score
- hyperparameters: `penalty` (either L1 or L2) and `C` between 0.01 and 3
- 8-fold CV

For both grid and randomized search, check 200 combinations of hyperparameters. Pick the right `solver` and `max_iter` parameters.

Print the best hyperparameters (`C` and `penalty`) for both `GridSearchCV` and `RandomizedSearchCV`. Are they similar?

Send the Jupyter notebook (with outputs) exported to `.html` format to lkrain@sgh.waw.pl or mruteck@sgh.waw.pl.